In [2]:
# Import pands lib to deal with data as tables
import pandas as pd

# To read vars values that stored in file .env
import os
from dotenv import load_dotenv

# To create connection with DB and view some information about them
from sqlalchemy import create_engine,inspect

# Load environment variables
load_dotenv("../.env")

# Set the vars with the responding values and read DB connection settings
db_user = os.getenv("DB_USER")
db_pass = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")

# Create database connection
engine = create_engine(
f"postgresql+psycopg2://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}"
    )

# Test the connection
with engine.connect() as connection:
    print("successfully connection with DB")

# Inspect available tables
inspector = inspect(engine)
tables = inspector.get_table_names()
print(tables)


successfully connection with DB
['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'product_category_name_translation']


In [3]:

# Tables required for the ML joining process
all_tables = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
    "product_category_name_translation"
]

# Read tables from PostgreSQL
data = {
    table: pd.read_sql(f'SELECT * FROM "{table}"', engine)
    for table in all_tables
}

# Review table dimensions and columns
for table, df in data.items():
    print(f"\n--- {table} ---")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())


--- customers ---
Shape: (99441, 5)
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

--- geolocation ---
Shape: (1000163, 6)
Columns: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state', 'geolocation_id']

--- order_items ---
Shape: (112650, 7)
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

--- order_payments ---
Shape: (103886, 5)
Columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

--- order_reviews ---
Shape: (99224, 7)
Columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

--- orders ---
Shape: (99441, 8)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_deliver

In [4]:
conclosion =[]

# for loop to set the data inside the list ,
for listable,dataf in data.items():
    #  set its values as dictionary
    conclosion.append({
        "tables":listable,
        "rows":len(dataf),
        "columns":len(dataf.columns),
        "dupli":dataf.duplicated().sum()
    })
# confarm the resulte in table view
conclosion_data =pd.DataFrame(conclosion)
conclosion_data

,tables,rows,columns,dupli
0,customers,99441,5,0
1,geolocation,1000163,6,0
2,order_items,112650,7,0
3,order_payments,103886,5,0
4,order_reviews,99224,7,0
5,orders,99441,8,0
6,products,32951,9,0
7,sellers,3095,4,0
8,product_category_name_translation,71,2,0


In [5]:
# to ensure the unique columnsfor each table
unique_col = {}
# for loop  
for listable, dataf in data.items():
    unique_col[listable] = [
        col
        for col in dataf.columns
        if dataf[col].nunique() == len(dataf)
    ]

unique_col            

{'customers': ['customer_id'],
 'geolocation': ['geolocation_id'],
 'order_items': [],
 'order_payments': [],
 'order_reviews': [],
 'orders': ['order_id', 'customer_id'],
 'products': ['product_id'],
 'sellers': ['seller_id'],
 'product_category_name_translation': ['product_category_name',
  'product_category_name_english']}

In [6]:
for table in ["orders", "order_items", "order_payments", "order_reviews"]:
    df = data[table]
    
    print(f"\n--- {table} ---")
    
    if "order_id" in df.columns:
        print("Rows:", len(df))
        print("Unique order_id:", df["order_id"].nunique())
        print("Duplicated order_id:", df["order_id"].duplicated().sum())


--- orders ---
Rows: 99441
Unique order_id: 99441
Duplicated order_id: 0

--- order_items ---
Rows: 112650
Unique order_id: 98666
Duplicated order_id: 13984

--- order_payments ---
Rows: 103886
Unique order_id: 99440
Duplicated order_id: 4446

--- order_reviews ---
Rows: 99224
Unique order_id: 98673
Duplicated order_id: 551


In [7]:
# check the real PK for all tables
pk_checks ={
    "customers" : ["customer_id"],
    "order_items" : ["order_id" , "order_item_id"],
    "order_payments" : ["order_id","payment_sequential"],
    "order_reviews": ["order_id","review_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    }
# for loop to check the combin is good or not
for rtable,columns in pk_checks.items():
    df = data[rtable]
    dup_count = df.duplicated(subset=columns).sum()
    print(f"{rtable}")
    print(f"pk columns:{columns}")
    print(f"dup combin:{dup_count}")


customers
pk columns:['customer_id']
dup combin:0
order_items
pk columns:['order_id', 'order_item_id']
dup combin:0
order_payments
pk columns:['order_id', 'payment_sequential']
dup combin:0
order_reviews
pk columns:['order_id', 'review_id']
dup combin:0
orders
pk columns:['order_id']
dup combin:0
products
pk columns:['product_id']
dup combin:0
sellers
pk columns:['seller_id']
dup combin:0


In [8]:
# test the FK between orders and customers
orders = data["orders"]
customers = data["customers"]
# check if there is any missing data
miss_data= ~ orders["customer_id"].isin(customers["customer_id"])
print("number of customer_id missing  is :",miss_data.sum())

number of customer_id missing  is : 0


In [9]:
# test the FK between orders and order items
orders = data["orders"]
order_items = data["order_items"]
# check if there is any missing data
miss_data= ~ order_items["order_id"].isin(orders["order_id"])
print("number of order_id missing  is :",miss_data.sum())

number of order_id missing  is : 0


In [10]:
# test the FK between products and order items
products = data["products"]
order_items = data["order_items"]
# check if there is any missing data
miss_data= ~ order_items["product_id"].isin(products["product_id"])
print("number of product_id missing  is :",miss_data.sum())

number of product_id missing  is : 0


In [11]:
# test the FK between sellers and order items
sellers = data["sellers"]
order_items = data["order_items"]
# check if there is any missing data
miss_data= ~ order_items["seller_id"].isin(sellers["seller_id"])
print("number of seller_id missing  is :",miss_data.sum())

number of seller_id missing  is : 0


In [12]:
# test the FK between order_reviews and orders
order_reviews = data["order_reviews"]
orders = data["orders"]
miss_data = ~order_reviews["order_id"].isin(orders["order_id"])
print(f"number of order_id missing  is :",miss_data.sum())

number of order_id missing  is : 0


In [13]:
# test the FK between order_payments and orders
order_payments = data["order_payments"]
orders = data["orders"]
miss_data = ~order_payments["order_id"].isin(orders["order_id"])
print(f"number of order_id missing  is :",miss_data.sum())

number of order_id missing  is : 0


In [14]:
#for loop to ensure that we have cover all the FK between all tables
for first_table, df1 in data.items():
    for sec_table, df2 in data.items():
        if first_table < sec_table:
            shered_col = set(df1.columns).intersection(df2.columns)

            if shered_col:
                print(f"{first_table}{sec_table}: {sorted(shered_col)}")

customersorders: ['customer_id']
order_itemsorder_payments: ['order_id']
order_itemsorder_reviews: ['order_id']
order_itemsorders: ['order_id']
order_itemsproducts: ['product_id']
order_itemssellers: ['seller_id']
order_paymentsorder_reviews: ['order_id']
order_paymentsorders: ['order_id']
order_reviewsorders: ['order_id']
product_category_name_translationproducts: ['product_category_name']


In [15]:
# know the number of rows that include 1 order per row or number of rows that include 1 order 
for table in ["order_items","order_payments","order_reviews"]:
    store = data[table]
    size_counter = store.groupby("order_id").size()

    print(f"\n ***{table}***")
    print("the represinted orders  :", size_counter.size)
    print("the max row include each order  :", size_counter.max())
    print("the avg row include each order :", size_counter.mean())
    print("the orders with more than one row :", (size_counter>1).sum())
         



 ***order_items***
the represinted orders  : 98666
the max row include each order  : 21
the avg row include each order : 1.1417306873695092
the orders with more than one row : 9803

 ***order_payments***
the represinted orders  : 99440
the max row include each order  : 29
the avg row include each order : 1.0447103781174578
the orders with more than one row : 2961

 ***order_reviews***
the represinted orders  : 98673
the max row include each order  : 3
the avg row include each order : 1.0055841010205426
the orders with more than one row : 547


In [16]:
# know the colomns name for tables before join the to assign the right colmns for aggregation
for table in ["order_items","order_payments","order_reviews"]:
    
    print(f"\n ***{table}***")
    # show the columns names in list 
    print(data[table].columns.tolist())


 ***order_items***
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

 ***order_payments***
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

 ***order_reviews***
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


In [17]:
# featurs build for order_items
order_items_features = (
    data["order_items"]
    .groupby("order_id")
    .agg(
        item_num =("order_item_id","count"),
        total_cost =("price","sum"),
        total_freight_cost=("freight_value","sum")
        )
.reset_index()
)

print(order_items_features.head())
print(order_items_features.shape)
print(order_items_features["order_id"].nunique())

                           order_id  item_num  total_cost  total_freight_cost
0  00010242fe8c5a6d1ba2dd792cb16214         1       58.90               13.29
1  00018f77f2f0320c557190d7a144bdd3         1      239.90               19.93
2  000229ec398224ef6ca0657da4fc703e         1      199.00               17.87
3  00024acbcdf0a6daa1e931b038114c75         1       12.99               12.79
4  00042b26cf59d7ce69dfabb4e55b4fd9         1      199.90               18.14
(98666, 4)
98666


In [18]:
# featurs build for order_payments
order_payments_features = (
    data["order_payments"]
    .groupby("order_id")
    .agg(
        payment_count=("payment_sequential", "count"),
        total_payment_value=("payment_value", "sum")
    )
    .reset_index()
)

print(order_payments_features.head())
print("\nShape:", order_payments_features.shape)
print("Unique orders:", order_payments_features["order_id"].nunique())

                           order_id  payment_count  total_payment_value
0  00010242fe8c5a6d1ba2dd792cb16214              1                72.19
1  00018f77f2f0320c557190d7a144bdd3              1               259.83
2  000229ec398224ef6ca0657da4fc703e              1               216.87
3  00024acbcdf0a6daa1e931b038114c75              1                25.78
4  00042b26cf59d7ce69dfabb4e55b4fd9              1               218.04

Shape: (99440, 3)
Unique orders: 99440


In [19]:
items = data["order_items"]

shipping_check = (
    items.groupby("order_id")["shipping_limit_date"]
    .agg(["count", "nunique"])
)

print(shipping_check.head())
print("\nOrders with more than one shipping limit date:",
      (shipping_check["nunique"] > 1).sum())

print("\nMissing shipping limit dates:",
      items["shipping_limit_date"].isna().sum())

                                  count  nunique
order_id                                        
00010242fe8c5a6d1ba2dd792cb16214      1        1
00018f77f2f0320c557190d7a144bdd3      1        1
000229ec398224ef6ca0657da4fc703e      1        1
00024acbcdf0a6daa1e931b038114c75      1        1
00042b26cf59d7ce69dfabb4e55b4fd9      1        1

Orders with more than one shipping limit date: 336

Missing shipping limit dates: 0


In [20]:
# try to know what columns to take
orders = data["orders"]

print("Shape:", orders.shape)
print("\nColumns:")
print(orders.columns.tolist())

print("\nFirst 5 rows:")
display(orders.head())

Shape: (99441, 8)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

First 5 rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [21]:

# 1. Count missing values in each column
print("Missing values per column:")
display(orders.isna().sum().to_frame(name="missing_count"))

# 2. Check duplicate order IDs
print("Duplicate order_id rows:", orders["order_id"].duplicated().sum())

# 3. Check missing values in important columns
print("Rows with missing order_id:", orders["order_id"].isna().sum())
print("Rows with missing customer_id:", orders["customer_id"].isna().sum())
print("Rows with missing order_status:", orders["order_status"].isna().sum())

Missing values per column:


,missing_count
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


Duplicate order_id rows: 0
Rows with missing order_id: 0
Rows with missing customer_id: 0
Rows with missing order_status: 0


In [22]:
# Count missing values in delivery-related columns by order status

missing_by_status = (
    orders.groupby("order_status")[
        [
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
        ]
    ]
    .apply(lambda group: group.isna().sum())
)

missing_by_status

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [23]:
# Display the columns and data types of the orders table

print("Columns:")
print(orders.columns.tolist())

print("\nData types:")
print(orders.dtypes)

print("\nShape:")
print(orders.shape)

print("\nUnique order IDs:")
print(orders["order_id"].nunique())

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Data types:
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date            object
dtype: object

Shape:
(99441, 8)

Unique order IDs:
99441


In [24]:
# Convert the estimated delivery date to datetime

orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"],
    errors="coerce"
)

# Verify the conversion

print(orders["order_estimated_delivery_date"].dtype)

print(
    "Missing estimated delivery dates:",
    orders["order_estimated_delivery_date"].isna().sum()
)

datetime64[ns]
Missing estimated delivery dates: 0


In [25]:
# Inspect the aggregated feature tables

feature_tables = {
    "order_items_features": order_items_features,
    "order_payments_features": order_payments_features,
}

for table_name, df in feature_tables.items():
    print(f"\n--- {table_name} ---")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Unique order IDs:", df["order_id"].nunique())
    print("Missing order IDs:", df["order_id"].isna().sum())
    print("Duplicate order IDs:", df["order_id"].duplicated().sum())


--- order_items_features ---


Shape: (98666, 4)
Columns: ['order_id', 'item_num', 'total_cost', 'total_freight_cost']
Unique order IDs: 98666
Missing order IDs: 0
Duplicate order IDs: 0

--- order_payments_features ---
Shape: (99440, 3)
Columns: ['order_id', 'payment_count', 'total_payment_value']
Unique order IDs: 99440
Missing order IDs: 0
Duplicate order IDs: 0


In [26]:
orders_base = orders[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_estimated_delivery_date"
    ]
].copy()

print("Shape:", orders_base.shape)
print("Unique order IDs:", orders_base["order_id"].nunique())
print("Duplicate order IDs:", orders_base["order_id"].duplicated().sum())
print("Missing values:\n", orders_base.isna().sum())

orders_base.head()

Shape: (99441, 5)
Unique order IDs: 99441
Duplicate order IDs: 0
Missing values:
 order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_estimated_delivery_date    0
dtype: int64


,order_id,customer_id,order_status,order_purchase_timestamp,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-26


In [27]:
# featurs order_items chek
print("Shape:", order_items_features.shape)
print("Columns:", order_items_features.columns.tolist())
print("Unique order IDs:", order_items_features["order_id"].nunique())
print("Duplicate order IDs:", order_items_features["order_id"].duplicated().sum())
print("Missing values:\n", order_items_features.isna().sum())

order_items_features.head()

Shape: (98666, 4)
Columns: ['order_id', 'item_num', 'total_cost', 'total_freight_cost']
Unique order IDs: 98666
Duplicate order IDs: 0
Missing values:
 order_id              0
item_num              0
total_cost            0
total_freight_cost    0
dtype: int64


,order_id,item_num,total_cost,total_freight_cost
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14


In [28]:
# first join code for orders_base and order_items
orders_joined = orders_base.merge(
    order_items_features,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Shape:", orders_joined.shape)
print("Unique order IDs:", orders_joined["order_id"].nunique())
print("Duplicate order IDs:", orders_joined["order_id"].duplicated().sum())
print("Missing values:\n", orders_joined.isna().sum())

orders_joined.head()

Shape: (99441, 8)
Unique order IDs: 99441
Duplicate order IDs: 0
Missing values:
 order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_estimated_delivery_date      0
item_num                         775
total_cost                       775
total_freight_cost               775
dtype: int64


,order_id,customer_id,order_status,order_purchase_timestamp,order_estimated_delivery_date,item_num,total_cost,total_freight_cost
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-18,1.0,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-13,1.0,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-09-04,1.0,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-15,1.0,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-26,1.0,19.90,8.72


In [29]:
# join code for orders_joined and order_payments_features
orders_joined = orders_joined.merge(
    order_payments_features,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Shape:", orders_joined.shape)
print("Unique order IDs:", orders_joined["order_id"].nunique())
print("Duplicate order IDs:", orders_joined["order_id"].duplicated().sum())
print("Missing values:\n", orders_joined.isna().sum())

orders_joined.head()

Shape: (99441, 10)
Unique order IDs: 99441
Duplicate order IDs: 0
Missing values:
 order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_estimated_delivery_date      0
item_num                         775
total_cost                       775
total_freight_cost               775
payment_count                      1
total_payment_value                1
dtype: int64


,order_id,customer_id,order_status,order_purchase_timestamp,order_estimated_delivery_date,item_num,total_cost,total_freight_cost,payment_count,total_payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-18,1.0,29.99,8.72,3.0,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-13,1.0,118.70,22.76,1.0,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-09-04,1.0,159.90,19.22,1.0,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-15,1.0,45.00,27.20,1.0,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-26,1.0,19.90,8.72,1.0,28.62


In [30]:
# customers table columns check
print("Shape:", customers.shape)
print("Columns:", customers.columns.tolist())
print("Unique customer IDs:", customers["customer_id"].nunique())
print("Duplicate customer IDs:", customers["customer_id"].duplicated().sum())
print("Missing values:\n", customers.isna().sum())

customers.head()

Shape: (99441, 5)
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
Unique customer IDs: 99441
Duplicate customer IDs: 0
Missing values:
 customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [31]:
# create a copy table from orgin customers table the join it with orders_joined
customers_features = customers[
    [
        "customer_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]
].copy()

orders_joined = orders_joined.merge(
    customers_features,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", orders_joined.shape)
print("Unique order IDs:", orders_joined["order_id"].nunique())
print("Duplicate order IDs:", orders_joined["order_id"].duplicated().sum())
print("Missing values:\n", orders_joined.isna().sum())

orders_joined.head()

Shape: (99441, 13)
Unique order IDs: 99441
Duplicate order IDs: 0
Missing values:
 order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_estimated_delivery_date      0
item_num                         775
total_cost                       775
total_freight_cost               775
payment_count                      1
total_payment_value                1
customer_zip_code_prefix           0
customer_city                      0
customer_state                     0
dtype: int64


,order_id,customer_id,order_status,order_purchase_timestamp,order_estimated_delivery_date,item_num,total_cost,total_freight_cost,payment_count,total_payment_value,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-18,1.0,29.99,8.72,3.0,38.71,03149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-13,1.0,118.70,22.76,1.0,141.46,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-09-04,1.0,159.90,19.22,1.0,179.12,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-15,1.0,45.00,27.20,1.0,72.20,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-26,1.0,19.90,8.72,1.0,28.62,09195,santo andre,SP


In [32]:
# الطلبات التي لا توجد لها بيانات منتجات
missing_items = orders_joined[
    orders_joined["item_num"].isna()
]

print("Orders without item features:", len(missing_items))
print("\nOrder status:")
print(missing_items["order_status"].value_counts(dropna=False))


# الطلبات التي لا توجد لها بيانات دفع
missing_payments = orders_joined[
    orders_joined["payment_count"].isna()
]

print("\nOrders without payment features:", len(missing_payments))
print("\nOrder status:")
print(missing_payments["order_status"].value_counts(dropna=False))

Orders without item features: 775

Order status:
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

Orders without payment features: 1

Order status:
order_status
delivered    1
Name: count, dtype: int64


In [33]:
# Order status check
print("Order status counts:")
print(orders_joined["order_status"].value_counts(dropna=False))

print("\nTotal orders:", len(orders_joined))
print(
    "Unique orders:",
    orders_joined["order_id"].nunique()
)

Order status counts:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Total orders: 99441
Unique orders: 99441


In [34]:
print("Shape:", products.shape)

print("\nColumns:")
print(products.columns.tolist())

print("\nMissing values:")
print(products.isna().sum())

products.head()

Shape: (32951, 9)

Columns:
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Missing values:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [35]:
# عدد المنتجات المختلفة في جدول المنتجات
print("Unique product IDs in products:",
      products["product_id"].nunique())

# عدد المنتجات المختلفة في عناصر الطلبات
print("Unique product IDs in order_items:",
      order_items["product_id"].nunique())

# عدد product_id في order_items غير الموجودة في products
missing_product_ids = (
    ~order_items["product_id"].isin(products["product_id"])
)

print("Order item rows without matching product:",
      missing_product_ids.sum())

Unique product IDs in products: 32951
Unique product IDs in order_items: 32951
Order item rows without matching product: 0


In [36]:
# اختيار الأعمدة التي نحتاجها من جدول المنتجات
products_features = products[
    [
        "product_id",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].copy()

# ربط عناصر الطلبات بخصائص المنتجات
order_items_products = order_items.merge(
    products_features,
    on="product_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", order_items_products.shape)
print("\nMissing values in product measurements:")
print(
    order_items_products[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isna().sum()
)

Shape: (112650, 11)

Missing values in product measurements:
product_weight_g     18
product_length_cm    18
product_height_cm    18
product_width_cm     18
dtype: int64


In [37]:
# حساب حجم كل منتج بالسنتيمتر المكعب
order_items_products["product_volume_cm3"] = (
    order_items_products["product_length_cm"]
    * order_items_products["product_height_cm"]
    * order_items_products["product_width_cm"]
)

# تلخيص خصائص المنتجات على مستوى الطلب
order_products_features = (
    order_items_products
    .groupby("order_id")
    .agg(
        total_weight_g=("product_weight_g", "sum"),
        avg_product_weight_g=("product_weight_g", "mean"),
        total_volume_cm3=("product_volume_cm3", "sum"),
        products_count=("product_id", "count"),
        products_missing_measurements=(
            "product_weight_g",
            lambda s: s.isna().sum()
        )
    )
    .reset_index()
)

print("Shape:", order_products_features.shape)
print("\nDuplicate order IDs:",
      order_products_features["order_id"].duplicated().sum())
print("\nMissing values:")
print(order_products_features.isna().sum())

order_products_features.head()

Shape: (98666, 6)

Duplicate order IDs: 0

Missing values:
order_id                          0
total_weight_g                    0
avg_product_weight_g             16
total_volume_cm3                  0
products_count                    0
products_missing_measurements     0
dtype: int64


,order_id,total_weight_g,avg_product_weight_g,total_volume_cm3,products_count,products_missing_measurements
0,00010242fe8c5a6d1ba2dd792cb16214,650.0,650.0,3528.0,1,0
1,00018f77f2f0320c557190d7a144bdd3,30000.0,30000.0,60000.0,1,0
2,000229ec398224ef6ca0657da4fc703e,3050.0,3050.0,14157.0,1,0
3,00024acbcdf0a6daa1e931b038114c75,200.0,200.0,2400.0,1,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,3750.0,3750.0,42000.0,1,0


In [38]:
# تحديد الطلبات التي ينقصها الوزن في جميع المنتجات
all_weights_missing = (
    order_products_features["products_missing_measurements"]
    == order_products_features["products_count"]
)

# جعل المجاميع غير معروفة بدلًا من اعتبارها صفرًا
order_products_features.loc[
    all_weights_missing,
    ["total_weight_g", "total_volume_cm3"]
] = float("nan")

print("Orders with all product measurements missing:",
      all_weights_missing.sum())

print("\nMissing values after correction:")
print(order_products_features.isna().sum())

Orders with all product measurements missing: 16

Missing values after correction:
order_id                          0
total_weight_g                   16
avg_product_weight_g             16
total_volume_cm3                 16
products_count                    0
products_missing_measurements     0
dtype: int64


In [39]:
# اضافة خصائص المنتجات إلى جدول الطلبات مع الحفاظ على صف واحد لكل طلب

orders_joined = orders_joined.merge(
    order_products_features,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Shape:", orders_joined.shape)

print("\nUnique orders:",
      orders_joined["order_id"].nunique())

print("Duplicate order IDs:",
      orders_joined["order_id"].duplicated().sum())

print("\nMissing values:")
print(orders_joined.isna().sum())

Shape: (99441, 18)

Unique orders: 99441
Duplicate order IDs: 0

Missing values:
order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_estimated_delivery_date      0
item_num                         775
total_cost                       775
total_freight_cost               775
payment_count                      1
total_payment_value                1
customer_zip_code_prefix           0
customer_city                      0
customer_state                     0
total_weight_g                   791
avg_product_weight_g             791
total_volume_cm3                 791
products_count                   775
products_missing_measurements    775
dtype: int64


In [40]:
print("Current columns:")
for column in orders_joined.columns:
    print("-", column)

Current columns:
- order_id
- customer_id
- order_status
- order_purchase_timestamp
- order_estimated_delivery_date
- item_num
- total_cost
- total_freight_cost
- payment_count
- total_payment_value
- customer_zip_code_prefix
- customer_city
- customer_state
- total_weight_g
- avg_product_weight_g
- total_volume_cm3
- products_count
- products_missing_measurements


In [41]:
print("Purchase timestamp range:")
print("Earliest:", orders_joined["order_purchase_timestamp"].min())
print("Latest:", orders_joined["order_purchase_timestamp"].max())

print("\nEstimated delivery date range:")
print("Earliest:", orders_joined["order_estimated_delivery_date"].min())
print("Latest:", orders_joined["order_estimated_delivery_date"].max())

# فحص الطلبات التي موعد تسليمها المتوقع يسبق تاريخ الشراء
invalid_estimated_dates = (
    orders_joined["order_estimated_delivery_date"]
    < orders_joined["order_purchase_timestamp"]
)

print(
    "\nOrders with estimated delivery before purchase:",
    invalid_estimated_dates.sum()
)

Purchase timestamp range:
Earliest: 2016-09-04 21:15:19
Latest: 2018-10-17 17:30:18

Estimated delivery date range:
Earliest: 2016-09-30 00:00:00
Latest: 2018-11-12 00:00:00

Orders with estimated delivery before purchase: 0


In [42]:
# إنشاء جدول منفصل يحتوي على تواريخ التسليم الفعلية
delivery_dates = orders[
    [
        "order_id",
        "order_delivered_customer_date"
    ]
].copy()

print("Shape:", delivery_dates.shape)
print("\nMissing actual delivery dates:")
print(delivery_dates["order_delivered_customer_date"].isna().sum())

print("\nData type:")
print(delivery_dates["order_delivered_customer_date"].dtype)

Shape: (99441, 2)

Missing actual delivery dates:
2965

Data type:
datetime64[ns]


In [43]:

# ننسخ الأعمدة التي نحتاجها لفحص حالات الطلبات وتواريخ التسليم
delivery_check = orders[
    [
        "order_id",
        "order_status",
        "order_estimated_delivery_date",
        "order_delivered_customer_date"
    ]
].copy()

# عدد الطلبات حسب الحالة
print("Order status distribution:")
print(delivery_check["order_status"].value_counts())

# نحدد الطلبات التي تم تسليمها فعليًا
delivered_orders = delivery_check[
    delivery_check["order_status"] == "delivered"
]

print("\nDelivered orders:", len(delivered_orders))

# نفحص تواريخ التسليم الفعلية داخل الطلبات المسلّمة
print(
    "Delivered orders with missing actual delivery date:",
    delivered_orders["order_delivered_customer_date"].isna().sum()
)

# نفحص تواريخ التسليم المتوقعة داخل الطلبات المسلّمة
print(
    "Delivered orders with missing estimated delivery date:",
    delivered_orders["order_estimated_delivery_date"].isna().sum()
)

Order status distribution:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Delivered orders: 96478
Delivered orders with missing actual delivery date: 8
Delivered orders with missing estimated delivery date: 0


In [44]:

# عرض الطلبات المسلّمة التي ينقصها تاريخ التسليم الفعلي
missing_delivery_dates = delivered_orders[
    delivered_orders["order_delivered_customer_date"].isna()
]

print(missing_delivery_dates.to_string(index=False))

                        order_id order_status order_estimated_delivery_date order_delivered_customer_date
2d1e2d5bf4dc7227b3bfebb81328c15f    delivered                    2017-12-18                           NaT
f5dd62b788049ad9fc0526e3ad11a097    delivered                    2018-07-16                           NaT
2ebdfc4f15f23b91474edf87475f108e    delivered                    2018-07-30                           NaT
e69f75a717d64fc5ecdfae42b2e8e086    delivered                    2018-07-30                           NaT
0d3268bad9b086af767785e3f0fc0133    delivered                    2018-07-24                           NaT
2d858f451373b04fb5c984a1cc2defaf    delivered                    2017-06-23                           NaT
ab7c89dc1bf4a1ead9d6ec1ec8968a84    delivered                    2018-06-26                           NaT
20edc82cf5400ce95e1afacc25798b31    delivered                    2018-07-19                           NaT


In [45]:

# نختار الطلبات المسلّمة التي يتوفر لها تاريخ التسليم الفعلي
delivered_with_dates = delivered_orders[
    delivered_orders["order_delivered_customer_date"].notna()
].copy()

# نتحقق هل يوجد طلب تم تسليمه قبل تاريخ شرائه
delivered_with_dates = delivered_with_dates.merge(
    orders[["order_id", "order_purchase_timestamp"]],
    on="order_id",
    how="left",
    validate="one_to_one"
)

delivered_before_purchase = delivered_with_dates[
    delivered_with_dates["order_delivered_customer_date"]
    < delivered_with_dates["order_purchase_timestamp"]
]

print("Delivered orders with actual dates:", len(delivered_with_dates))
print("Delivered before purchase:", len(delivered_before_purchase))

print(
    delivered_before_purchase[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_customer_date"
        ]
    ].head(10).to_string(index=False)
)


Delivered orders with actual dates: 96470
Delivered before purchase: 0
Empty DataFrame
Columns: [order_id, order_purchase_timestamp, order_delivered_customer_date]
Index: []


In [46]:

# 1. التأكد من حجم جدول ML النهائي
print("Shape:", orders_joined.shape)

# 2. التأكد من أن كل طلب يظهر مرة واحدة
print("Unique order IDs:", orders_joined["order_id"].nunique())
print("Duplicate order IDs:", orders_joined["order_id"].duplicated().sum())

# 3. عرض أنواع الأعمدة
print("\nColumn data types:")
print(orders_joined.dtypes)

# 4. فحص القيم المفقودة في كل عمود
print("\nMissing values per column:")
print(orders_joined.isna().sum())

Shape: (99441, 18)
Unique order IDs: 99441
Duplicate order IDs: 0

Column data types:
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
item_num                                float64
total_cost                              float64
total_freight_cost                      float64
payment_count                           float64
total_payment_value                     float64
customer_zip_code_prefix                 object
customer_city                            object
customer_state                           object
total_weight_g                          float64
avg_product_weight_g                    float64
total_volume_cm3                        float64
products_count                          float64
products_missing_measurements           float64
dtype: object

Missing values per column:
order_id

In [49]:
import os

os.makedirs("../articles", exist_ok=True)

orders_joined.to_csv(
    "../articles/orders_joined.csv",
    index=False
)